# Water Cooling Calculation

## 1. Flow Rate Calculation

### US Units System

All calculations will use **US customary units** for consistency.

**Hazen-Williams Equation (US Units):**

$$S = \frac{\Delta P}{L} = \frac{4.52 \cdot Q^{1.852}}{C^{1.852} \cdot D^{4.87}}$$

Where:
- **S** = frictional resistance (pressure drop per foot) in **psi/ft**
- **ΔP** = pressure drop over length L in **psi** (pounds per square inch absolute)
- **L** = length of pipe in **feet**
- **Q** = flow rate in **gpm** (gallons per minute)
- **C** = pipe roughness coefficient (dimensionless)
- **D** = inside pipe diameter in **inches**

**For coil flow rate:** $Q_{coil} = \left(\frac{C^{1.852} \cdot d^{4.87} \cdot \Delta P_{coil}}{4.52 \cdot L}\right)^{0.526}$

**Alternative simplified calculation:** $Q = k \sqrt{\frac{\Delta P \cdot D^2}{L}}$ (with appropriate US unit constants)

## 2. Heat Transfer Calculations (SI Units)

### Simple Version: Water Temperature Rise

For basic calculations using SI units, we can estimate the water temperature rise using:

$$P_{heatload} = \dot{m} C_p \Delta T$$

Which simplifies to:

$$\Delta T_{°C} = \frac{P_{W}}{\dot{m}_{kg/s} \times C_p}$$

Or in practical units:

$$\Delta T_{°C} = \frac{P_{W} \times 60}{Q_{L/min} \times \rho \times C_p} = \frac{P_{W}}{Q_{L/min} \times 69.77}$$

Where:
- **P** = Heat Load in **W** (Watts)
- **Q** = Flow Rate in **L/min** (liters per minute)  
- **ρ** = Water density ≈ **998 kg/m³**
- **Cp** = Specific heat of water ≈ **4182 J/(kg·K)**
- **69.77** = conversion factor (ρ × Cp / 60) for L/min to mass flow

## 3. Calculation for Sn Coils

### 3.1 Flow Rate Calculator Class

The `FlowRateCalculator` class provides comprehensive hydraulic calculations for water cooling systems using **US customary units** (gpm, inches, feet, psi). This class includes multiple calculation methods to handle different engineering scenarios:

#### Key Methods:

1. **`hazen_williams_pipe_loss(Q_gpm, C, D_inch, L_ft)`**
   - Calculates pressure loss in pipes using the Hazen-Williams equation
   - Input: flow rate (gpm), roughness coefficient, diameter (inch), length (ft)
   - Output: pressure drop (psi)

2. **`coil_flow_rate_hazen_williams(delta_P_psi, C, D_inch, L_ft)`** 
   - Determines achievable flow rate given available pressure drop
   - Input: pressure drop (psi), roughness coefficient, diameter (inch), length (ft)
   - Output: flow rate (gpm)

3. **`ucsb_darcy_flow_rate(delta_P_psi, D_H_inch, d_inch, L_ft)`**
   - UCSB thesis method for Darcy-based flow calculations
   - Input: pressure drop (psi), hydraulic diameter (inch), pipe diameter (inch), length (ft)
   - Output: flow rate (gpm)

4. **`effective_diameter_inch(width_inch, height_inch)`**
   - Calculates effective diameter for rectangular cross-sections
   - Uses the hydraulic diameter formula: D_eff = 4 × Area / Perimeter
   - Input: width (inch), height (inch) of rectangular channel
   - Output: effective diameter (inch)

5. **`Water_temp_rise_C(P_W, Q_gpm)`**
   - Calculates water temperature rise due to heat load
   - Input: heat load in Watts, flow rate in gpm
   - Output: temperature rise in °C
   - Uses fundamental heat transfer equation: ΔT = P / (ṁ × Cp)

#### Usage Example:
```python
# Initialize calculator
flow_calc = FlowRateCalculator()

# Calculate pressure loss
pressure_drop = flow_calc.hazen_williams_pipe_loss(Q_gpm=5.0, C=130, D_inch=0.25, L_ft=10)

# Calculate achievable flow rate
flow_rate = flow_calc.coil_flow_rate_hazen_williams(delta_P_psi=2.0, C=130, D_inch=0.25, L_ft=10)

# Calculate effective diameter for rectangular channel
eff_diameter = flow_calc.effective_diameter_inch(width_inch=0.5, height_inch=0.25)

# Calculate water temperature rise
temp_rise = flow_calc.Water_temp_rise_C(P_W=1000, Q_gpm=2.0)  # 1kW heat load, 2 gpm flow
```

**Note:** All calculations use absolute pressure (psi) rather than gauge pressure (psig) for accuracy.

### 3.2 Flow Rate Calculator Implementation


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import fsolve

class FlowRateCalculator:
    """
    Calculator for water cooling flow rates using US customary units
    All calculations use: gpm, inches, feet, psi
    """
    
    def __init__(self):
        # FlowRateCalculator focuses on hydraulic calculations using US units
        # Water properties for temperature rise calculations
        self.rho_kg_m3 = 998  # Water density at 20°C (kg/m³)
        self.Cp_J_kg_K = 4182  # Specific heat of water (J/(kg·K))
    
    def hazen_williams_pipe_loss(self, Q_gpm, C, D_inch, L_ft):
        """
        Calculate pressure loss in pipe using Hazen-Williams equation (US units)
        
        Parameters:
        Q_gpm: Flow rate in gallons per minute (gpm)
        C: Roughness coefficient (typically 130-150 for smooth pipes, 120 for steel, 100 for old pipes)
        D_inch: Pipe diameter in inches
        L_ft: Pipe length in feet
        
        Returns:
        Pressure loss in psi (pounds per square inch absolute)
        """
        # Hazen-Williams equation (US units): S = 4.52 * Q^1.852 / (C^1.852 * D^4.87)
        # Where S is pressure drop per foot (psi/ft)
        S = 4.52 * (Q_gpm**1.852) / (C**1.852 * D_inch**4.87)
        delta_P_psi = S * L_ft  # Total pressure drop
        return delta_P_psi
    
    def coil_flow_rate_hazen_williams(self, delta_P_psi, C, D_inch, L_ft):
        """
        Calculate flow rate in coil using Hazen-Williams equation (US units)
        
        Parameters:
        delta_P_psi: Available pressure drop for coil in psi
        C: Roughness coefficient
        D_inch: Coil diameter in inches
        L_ft: Coil length in feet
        
        Returns:
        Flow rate in gpm
        """
        # Rearranged Hazen-Williams: Q = (S * C^1.852 * D^4.87 / 4.52)^(1/1.852)
        # Where S = delta_P / L
        S = delta_P_psi / L_ft  # Pressure drop per foot
        Q_gpm = ((S * C**1.852 * D_inch**4.87) / 4.52)**(1/1.852)
        return Q_gpm
    
    
    def ucsb_darcy_flow_rate(self, delta_P_psi, D_H_inch, d_inch, L_ft):
        """
        UCSB Thesis method - Darcy-based flow rate calculation (US units)
        Based on: Q = 8000 * sqrt(ΔP * D_H * d² / L) (original in metric)
        Converted to US units
        
        Parameters:
        delta_P_psi: Pressure drop in psi
        D_H_inch: Hydraulic diameter in inches
        d_inch: Pipe diameter in inches
        L_ft: Length in feet
        
        Returns:
        Flow rate in gpm
        """
        # Convert to metric for calculation, then back to US units
        # Original equation: Q = 8000 * sqrt(ΔP(atm) * D_H(cm) * d²(cm) / L(cm))
        delta_P_atm = delta_P_psi / 14.7  # Convert psi to atm
        D_H_cm = D_H_inch * 2.54  # Convert inches to cm
        d_cm = d_inch * 2.54
        L_cm = L_ft * 30.48  # Convert feet to cm
        
        # Calculate flow rate in the original units (L/min based on the constant 8000)
        Q_mls = 8000 * np.sqrt(delta_P_atm * D_H_cm  / L_cm)* d_cm**2
        Q_Lmin = Q_mls * 60 /1000
        
        # Convert mL/s to gpm
        Q_gpm = Q_Lmin * 0.2642
        return Q_gpm
    
    def effective_diameter_inch(self, width_inch, height_inch):
        """
        Calculate effective diameter for rectangular cross-sections (US units)
        
        Parameters:
        width_inch: Width in inches
        height_inch: Height in inches
        
        Returns:
        Effective diameter in inches
        """
        # For rectangular: D_eff = 4*Area/Perimeter
        area = width_inch * height_inch
        perimeter = 2 * (width_inch + height_inch)
        D_eff = 4 * area / perimeter
        return D_eff
    
    def Water_temp_rise_C(self, P_W, Q_gpm):
        """
        Calculate simple water temperature rise (SI units)
        
        Parameters:
        P_W: Heat load in Watts
        Q_gpm: Flow rate in gallons per minute (gpm)

        Returns:
        Temperature rise in °C
        """
        # Convert flow rate to kg/s: gpm → L/min → m³/s → kg/s
        Q_Lmin = Q_gpm * 3.785  # gpm to L/min
        Q_m3_s = Q_Lmin / (1000 * 60)  # L/min to m³/s
        mass_flow_kg_s = Q_m3_s * self.rho_kg_m3  # m³/s to kg/s

        # ΔT = P / (ṁ × Cp)
        delta_T_C = P_W / (mass_flow_kg_s * self.Cp_J_kg_K)
        return delta_T_C

    
    def unit_conversions(self):
        """
        Display common unit conversions for reference
        """
        print("UNIT CONVERSION REFERENCE:")
        print("-" * 40)
        print("Flow Rate:")
        print("  1 gpm = 3.785 L/min = 0.06309 L/s")
        print("  1 L/min = 0.2642 gpm")
        print()
        print("Length:")
        print("  1 foot = 0.3048 m = 12 inches")
        print("  1 inch = 25.4 mm = 0.0254 m")
        print("  1 meter = 3.281 ft = 39.37 inches")
        print()
        print("Pressure:")
        print("  1 psi = 6.895 kPa = 6895 Pa")
        print("  1 kPa = 0.145 psi")
        print("  1 Pa = 0.000145 psi")
        print("  1 atm = 14.7 psi = 101.325 kPa")
  

# Initialize calculator
flow_calc = FlowRateCalculator()
print("Flow rate calculator initialized !")

# Display unit conversions for reference
flow_calc.unit_conversions()

Flow rate calculator initialized !
UNIT CONVERSION REFERENCE:
----------------------------------------
Flow Rate:
  1 gpm = 3.785 L/min = 0.06309 L/s
  1 L/min = 0.2642 gpm

Length:
  1 foot = 0.3048 m = 12 inches
  1 inch = 25.4 mm = 0.0254 m
  1 meter = 3.281 ft = 39.37 inches

Pressure:
  1 psi = 6.895 kPa = 6895 Pa
  1 kPa = 0.145 psi
  1 Pa = 0.000145 psi
  1 atm = 14.7 psi = 101.325 kPa


### 3.2 MOT Coil
Assuming 50 psi

Heat_load_W = 300 W (500A)

In [34]:
## Coil Paramter:
available_pressure_psi = 50  # psi 
MOT_coil_roughness = 140  # Copper
MOT_Coil_diameter_inch = 0.1 # inches (Coil Inner Diameter)
MOT_Coil_length_inch = 277
MOT_Coil_length_ft = MOT_Coil_length_inch / 12 # feet (Coil_Length)
heat_load_W = 300  # Heat load in Watts
inlet_temp_C = 20.0  # Inlet water temperature in °C

In [35]:
print("1.MOT Coil Flow Rate Calculation (Hazen-Williams Equation):")
print("-" * 50)

MOT_coil_flow_rate_gpm = flow_calc.coil_flow_rate_hazen_williams(available_pressure_psi, MOT_coil_roughness, MOT_Coil_diameter_inch, MOT_Coil_length_ft)
print(f"Available pressure for coil: {available_pressure_psi:.2f} psi ({(available_pressure_psi-14.7)*6.895:.0f} kPa gauge)")
print(f"Coil: {MOT_Coil_diameter_inch}\" diameter, {MOT_Coil_length_ft} ft length")
print(f"Flow rate: {MOT_coil_flow_rate_gpm:.2f} gpm ({MOT_coil_flow_rate_gpm*3.785:.2f} L/min)")


# Example 3: Simplified flow rate calculation
print("\n2. Simplified Flow Rate Calculation (UCSB Thesis):")
print("-" * 50)

hydraulic_diameter_inch=MOT_Coil_diameter_inch# For circular pipe, D_H = D
MOT_flow_rate_ucsb_gpm = flow_calc.ucsb_darcy_flow_rate(available_pressure_psi, hydraulic_diameter_inch, MOT_Coil_diameter_inch, MOT_Coil_length_ft)
print(f"Pressure drop: {available_pressure_psi:.1f} psi")
print(f"Tube: {MOT_Coil_diameter_inch}\" diameter, {MOT_Coil_length_ft} ft length")
print(f"Flow rate : {MOT_flow_rate_ucsb_gpm:.2f} gpm ({MOT_flow_rate_ucsb_gpm*3.785:.2f} L/min)")

1.MOT Coil Flow Rate Calculation (Hazen-Williams Equation):
--------------------------------------------------
Available pressure for coil: 50.00 psi (243 kPa gauge)
Coil: 0.1" diameter, 23.083333333333332 ft length
Flow rate: 0.22 gpm (0.84 L/min)

2. Simplified Flow Rate Calculation (UCSB Thesis):
--------------------------------------------------
Pressure drop: 50.0 psi
Tube: 0.1" diameter, 23.083333333333332 ft length
Flow rate : 0.29 gpm (1.09 L/min)


In [42]:
MOT_Coil_flow_rate_Lmin = MOT_coil_flow_rate_gpm * 3.785  # Convert gpm to L/min

print("Water Temperature Rise Calculation:")
print("-" * 50)

temp_rise_C = flow_calc.Water_temp_rise_C(heat_load_W, MOT_Coil_flow_rate_Lmin)
outlet_temp_C = inlet_temp_C + temp_rise_C

print(f"Water temperature rise: {temp_rise_C:.2f}°C")
print(f"Outlet temperature: {outlet_temp_C:.2f}°C")

Water Temperature Rise Calculation:
--------------------------------------------------
Water temperature rise: 0.10°C
Outlet temperature: 20.10°C


### 3.3 Slowing Coil 1 (Extended version)

Assuming 50 psi 

Heat_load_power = 23W

In [43]:
## Coil Paramter:
available_pressure_psi = 50  # psi 
slowing_1_coil_roughness = 140  # Copper
slowing_1_Coil_diameter_inch = 0.1 # inches (Coil Inner Diameter)
slowing_1_Coil_length_inch = 345
slowing_1_Coil_length_ft = slowing_1_Coil_length_inch / 12 # feet (Coil_Length)
heat_load_W = 23  # Heat load in Watts
inlet_temp_C = 20.0  # Inlet water temperature in °C

In [53]:
print("1.Slowing Coil 1 Flow Rate Calculation (Hazen-Williams Equation):")
print("-" * 50)

slowing_1_coil_flow_rate_gpm = flow_calc.coil_flow_rate_hazen_williams(available_pressure_psi, slowing_1_coil_roughness, slowing_1_Coil_diameter_inch, slowing_1_Coil_length_ft)
print(f"Available pressure for coil: {available_pressure_psi:.2f} psi ({(available_pressure_psi-14.7)*6.895:.0f} kPa gauge)")
print(f"Coil: {slowing_1_Coil_diameter_inch}\" diameter, {slowing_1_Coil_length_ft} ft length")
print(f"Flow rate: {slowing_1_coil_flow_rate_gpm:.2f} gpm ({slowing_1_coil_flow_rate_gpm*3.785:.2f} L/min)")


# Example 3: Simplified flow rate calculation
print("\n2. Simplified Flow Rate Calculation (UCSB Thesis):")
print("-" * 50)

hydraulic_diameter_inch=slowing_1_Coil_diameter_inch# For circular pipe, D_H = D
slowing_1_flow_rate_ucsb_gpm = flow_calc.ucsb_darcy_flow_rate(available_pressure_psi, hydraulic_diameter_inch, slowing_1_Coil_diameter_inch, slowing_1_Coil_length_ft)
print(f"Pressure drop: {available_pressure_psi:.1f} psi")
print(f"Tube: {slowing_1_Coil_diameter_inch}\" diameter, {slowing_1_Coil_length_ft} ft length")
print(f"Flow rate : {slowing_1_flow_rate_ucsb_gpm:.2f} gpm ({slowing_1_flow_rate_ucsb_gpm*3.785:.2f} L/min)")

slowing_1_Coil_flow_rate_Lmin = slowing_1_coil_flow_rate_gpm * 3.785  # Convert gpm to L/min

print("\n1. Water Temperature Rise Calculation:")
print("-" * 50)

temp_rise_C = flow_calc.Water_temp_rise_C(heat_load_W, slowing_1_Coil_flow_rate_Lmin)
outlet_temp_C = inlet_temp_C + temp_rise_C

print(f"Water temperature rise: {temp_rise_C:.2f}°C")
print(f"Outlet temperature: {outlet_temp_C:.2f}°C")

1.Slowing Coil 1 Flow Rate Calculation (Hazen-Williams Equation):
--------------------------------------------------
Available pressure for coil: 50.00 psi (243 kPa gauge)
Coil: 0.1" diameter, 28.75 ft length
Flow rate: 0.20 gpm (0.74 L/min)

2. Simplified Flow Rate Calculation (UCSB Thesis):
--------------------------------------------------
Pressure drop: 50.0 psi
Tube: 0.1" diameter, 28.75 ft length
Flow rate : 0.26 gpm (0.97 L/min)

1. Water Temperature Rise Calculation:
--------------------------------------------------
Water temperature rise: 0.08°C
Outlet temperature: 20.08°C


### 3.4 Slowing Coil 2
Assuming 50 psi

Heat load 42W

In [46]:
## Coil Paramter:
available_pressure_psi = 50  # psi 
slowing_2_coil_roughness = 140  # Copper
slowing_2_Coil_diameter_inch = 0.1 # inches (Coil Inner Diameter)
slowing_2_Coil_length_inch = 630
slowing_2_Coil_length_ft = slowing_2_Coil_length_inch / 12 # feet (Coil_Length)
heat_load_W = 23  # Heat load in Watts
inlet_temp_C = 20.0  # Inlet water temperature in °C

In [47]:
print("1.Slowing Coil 2 Flow Rate Calculation (Hazen-Williams Equation):")
print("-" * 50)

slowing_2_coil_flow_rate_gpm = flow_calc.coil_flow_rate_hazen_williams(available_pressure_psi, slowing_2_coil_roughness, slowing_2_Coil_diameter_inch, slowing_2_Coil_length_ft)
print(f"Available pressure for coil: {available_pressure_psi:.2f} psi ({(available_pressure_psi-14.7)*6.895:.0f} kPa gauge)")
print(f"Coil: {slowing_2_Coil_diameter_inch}\" diameter, {slowing_2_Coil_length_ft} ft length")
print(f"Flow rate: {slowing_2_coil_flow_rate_gpm:.2f} gpm ({slowing_2_coil_flow_rate_gpm*3.785:.2f} L/min)")


# Example 3: Simplified flow rate calculation
print("\n2. Simplified Flow Rate Calculation (UCSB Thesis):")
print("-" * 50)

hydraulic_diameter_inch=slowing_2_Coil_diameter_inch# For circular pipe, D_H = D
slowing_2_flow_rate_ucsb_gpm = flow_calc.ucsb_darcy_flow_rate(available_pressure_psi, hydraulic_diameter_inch, slowing_2_Coil_diameter_inch, slowing_2_Coil_length_ft)
print(f"Pressure drop: {available_pressure_psi:.1f} psi")
print(f"Tube: {slowing_2_Coil_diameter_inch}\" diameter, {slowing_2_Coil_length_ft} ft length")
print(f"Flow rate : {slowing_2_flow_rate_ucsb_gpm:.2f} gpm ({slowing_2_flow_rate_ucsb_gpm*3.785:.2f} L/min)")

slowing_2_Coil_flow_rate_Lmin = slowing_2_coil_flow_rate_gpm * 3.785  # Convert gpm to L/min

print("\n1. Water Temperature Rise Calculation:")
print("-" * 50)

temp_rise_C = flow_calc.Water_temp_rise_C(heat_load_W, slowing_2_Coil_flow_rate_Lmin)
outlet_temp_C = inlet_temp_C + temp_rise_C

print(f"Water temperature rise: {temp_rise_C:.2f}°C")
print(f"Outlet temperature: {outlet_temp_C:.2f}°C")

1.Slowing Coil 2 Flow Rate Calculation (Hazen-Williams Equation):
--------------------------------------------------
Available pressure for coil: 50.00 psi (243 kPa gauge)
Coil: 0.1" diameter, 52.5 ft length
Flow rate: 0.14 gpm (0.54 L/min)

2. Simplified Flow Rate Calculation (UCSB Thesis):
--------------------------------------------------
Pressure drop: 50.0 psi
Tube: 0.1" diameter, 52.5 ft length
Flow rate : 0.19 gpm (0.72 L/min)

1. Water Temperature Rise Calculation:
--------------------------------------------------
Water temperature rise: 0.16°C
Outlet temperature: 20.16°C


### 3.5 Slowing Coil 3
Assuming 50 psi 

Heat load 16W

In [50]:
## Coil Paramter:
available_pressure_psi = 50  # psi 
slowing_3_coil_roughness = 140  # Copper
slowing_3_Coil_diameter_inch = 0.1 # inches (Coil Inner Diameter)
slowing_3_Coil_length_inch = 248
slowing_3_Coil_length_ft = slowing_3_Coil_length_inch / 12 # feet (Coil_Length)
heat_load_W = 16  # Heat load in Watts
inlet_temp_C = 20.0  # Inlet water temperature in °C

In [51]:
print("1.Slowing Coil 3 Flow Rate Calculation (Hazen-Williams Equation):")
print("-" * 50)

slowing_3_coil_flow_rate_gpm = flow_calc.coil_flow_rate_hazen_williams(available_pressure_psi, slowing_3_coil_roughness, slowing_3_Coil_diameter_inch, slowing_3_Coil_length_ft)
print(f"Available pressure for coil: {available_pressure_psi:.2f} psi ({(available_pressure_psi-14.7)*6.895:.0f} kPa gauge)")
print(f"Coil: {slowing_3_Coil_diameter_inch}\" diameter, {slowing_3_Coil_length_ft} ft length")
print(f"Flow rate: {slowing_3_coil_flow_rate_gpm:.2f} gpm ({slowing_3_coil_flow_rate_gpm*3.785:.2f} L/min)")


# Example 3: Simplified flow rate calculation
print("\n2. Simplified Flow Rate Calculation (UCSB Thesis):")
print("-" * 50)

hydraulic_diameter_inch=slowing_3_Coil_diameter_inch# For circular pipe, D_H = D
slowing_3_flow_rate_ucsb_gpm = flow_calc.ucsb_darcy_flow_rate(available_pressure_psi, hydraulic_diameter_inch, slowing_3_Coil_diameter_inch, slowing_3_Coil_length_ft)
print(f"Pressure drop: {available_pressure_psi:.1f} psi")
print(f"Tube: {slowing_3_Coil_diameter_inch}\" diameter, {slowing_3_Coil_length_ft} ft length")
print(f"Flow rate : {slowing_3_flow_rate_ucsb_gpm:.2f} gpm ({slowing_3_flow_rate_ucsb_gpm*3.785:.2f} L/min)")

slowing_3_Coil_flow_rate_Lmin = slowing_3_coil_flow_rate_gpm * 3.785  # Convert gpm to L/min

print("\n1. Water Temperature Rise Calculation:")
print("-" * 50)

temp_rise_C = flow_calc.Water_temp_rise_C(heat_load_W, slowing_3_Coil_flow_rate_Lmin)
outlet_temp_C = inlet_temp_C + temp_rise_C

print(f"Water temperature rise: {temp_rise_C:.2f}°C")
print(f"Outlet temperature: {outlet_temp_C:.2f}°C")

1.Slowing Coil 3 Flow Rate Calculation (Hazen-Williams Equation):
--------------------------------------------------
Available pressure for coil: 50.00 psi (243 kPa gauge)
Coil: 0.1" diameter, 20.666666666666668 ft length
Flow rate: 0.23 gpm (0.89 L/min)

2. Simplified Flow Rate Calculation (UCSB Thesis):
--------------------------------------------------
Pressure drop: 50.0 psi
Tube: 0.1" diameter, 20.666666666666668 ft length
Flow rate : 0.30 gpm (1.15 L/min)

1. Water Temperature Rise Calculation:
--------------------------------------------------
Water temperature rise: 0.07°C
Outlet temperature: 20.07°C
